# 🎓 Лабораторна робота №3
## Методи інтелектуальної обробки даних
### Тема: Лінійна регресія та попередня обробка даних

---

**Датасет:** California Housing (sklearn)  
**Мета:** побудувати та порівняти лінійні регресійні моделі з різними видами попередньої обробки даних.

## 📦 Імпорт бібліотек

Використовуємо стандартний стек для машинного навчання:
- `numpy` / `pandas` — обробка даних
- `matplotlib` / `seaborn` — візуалізація
- `sklearn` — моделі, метрики, препроцесинг

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_log_error,
    max_error,
    d2_absolute_error_score
)
from sklearn.datasets import fetch_california_housing

print('✅ Усі бібліотеки успішно імпортовано')

---
## 🗂️ ЗАВДАННЯ 1. ПІДГОТОВКА НАБОРУ ДАНИХ

### 1.1 Завантаження датасету

**California Housing** — класичний датасет для задач регресії.  
Містить дані про 20 640 блоків житлових будинків Каліфорнії (перепис 1990 р.).

| Ознака | Опис |
|--------|------|
| `MedInc` | Медіанний дохід домогосподарств (у $10 000) |
| `HouseAge` | Медіанний вік будинків у блоці |
| `AveRooms` | Середня кількість кімнат |
| `AveBedrms` | Середня кількість спалень |
| `Population` | Населення блоку |
| `AveOccup` | Середня кількість мешканців |
| `Latitude` / `Longitude` | Координати |
| **`MedHouseVal`** | 🎯 **Цільова змінна** — медіанна вартість житла (у $100 000) |

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print(f'Розмір датасету: {df.shape[0]} рядків × {df.shape[1]} колонок\n')
df.head()

In [ ]:
# Базова статистика
df.describe().round(2)

### 1.2 Базова підготовка даних

**Кроки очистки:**
1. Перевірка типів та пропусків
2. Видалення аномалій методом **IQR** із `factor=3.0`

> 💡 **Чому `factor=3.0`?**  
> Стандартний IQR-фільтр використовує `factor=1.5`. Значення `3.0` є м'якшим — видаляємо лише екстремальні викиди, зберігаючи природну варіативність даних про нерухомість.

In [ ]:
FEATURE_COLS = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                'Population', 'AveOccup', 'Latitude', 'Longitude']
TARGET_COL   = 'MedHouseVal'

df = df[FEATURE_COLS + [TARGET_COL]]

print('▶ Пропущені значення:')
print(df.isnull().sum())
print('\n✅ Пропуски відсутні')

In [ ]:
def remove_outliers_iqr(dataframe, cols, factor=3.0):
    """
    Видалення аномалій методом IQR.
    factor=3.0 — м'який фільтр, видаляє лише екстремальні викиди.
    Межі: [Q1 - 3*IQR, Q3 + 3*IQR]
    """
    mask = pd.Series([True] * len(dataframe), index=dataframe.index)
    for col in cols:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        mask &= (dataframe[col] >= lower) & (dataframe[col] <= upper)
    return dataframe[mask]

all_cols = FEATURE_COLS + [TARGET_COL]
df_clean = remove_outliers_iqr(df, all_cols)

removed = len(df) - len(df_clean)
print(f'Видалено аномалій: {removed} рядків ({removed/len(df)*100:.1f}%)')
print(f'Залишилось: {df_clean.shape[0]} рядків')

In [ ]:
# Візуалізація розподілу цільової змінної до і після очистки
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[TARGET_COL], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('До очистки (з аномаліями)', fontsize=13)
axes[0].set_xlabel('MedHouseVal')
axes[0].set_ylabel('Частота')
axes[0].axvline(df[TARGET_COL].median(), color='red', linestyle='--', label=f'Медіана={df[TARGET_COL].median():.2f}')
axes[0].legend()

axes[1].hist(df_clean[TARGET_COL], bins=50, color='seagreen', edgecolor='white', alpha=0.8)
axes[1].set_title('Після очистки (IQR, factor=3.0)', fontsize=13)
axes[1].set_xlabel('MedHouseVal')
axes[1].axvline(df_clean[TARGET_COL].median(), color='red', linestyle='--', label=f'Медіана={df_clean[TARGET_COL].median():.2f}')
axes[1].legend()

plt.suptitle('Розподіл цільової змінної (MedHouseVal)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Верхнє обрізання (кластер біля 5.0) видалено — ціни, що впираються в стелю датасету.')

### 1.3 Аналіз залежностей та відбір ознак

**Метод:** матриця Пірсонівської кореляції.

**Логіка відбору:**
- Ознаки **сортуємо за |r| з цільовою змінною** — найбільш інформативні йдуть першими.
- Якщо дві ознаки корелюють між собою сильніше за `0.85` — залишаємо ту, що краще пояснює ціну.

In [ ]:
corr_matrix = df_clean.corr()

# Ознаки відсортовані за силою зв'язку з цільовою змінною
target_corr = corr_matrix[TARGET_COL].drop(TARGET_COL).abs().sort_values(ascending=False)
sorted_features = target_corr.index.tolist()
BEST_FEATURE = sorted_features[0]

print(f'▶ Кореляція ознак з "{TARGET_COL}" (за спаданням абсолютного значення):\n')
corr_display = corr_matrix[TARGET_COL].drop(TARGET_COL).reindex(sorted_features)
for feat, val in corr_display.items():
    bar = '█' * int(abs(val) * 20)
    sign = '+' if val > 0 else '-'
    print(f'  {feat:<12} {sign}{bar:<20} {val:+.4f}')

print(f'\n🏆 Найбільш корельована ознака: {BEST_FEATURE}')

In [ ]:
# Теплова карта кореляцій
order = sorted_features + [TARGET_COL]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix.loc[order, order],
    annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, ax=ax, center=0,
    vmin=-1, vmax=1
)
ax.set_title('Теплова карта кореляцій\n(ознаки відсортовано за зв\'язком з цільовою змінною)', fontsize=13)
plt.tight_layout()
plt.show()

print('💡 Зверніть увагу: MedInc має найсильніший позитивний зв\'язок з ціною.')
print('   Latitude/Longitude мають помітний від\'ємний зв\'язок — північ Каліфорнії дешевше.')

In [ ]:
# Видалення сильно взаємокорельованих ознак (|r| > 0.85)
CORR_THRESHOLD = 0.85
drop_features = set()
feature_corr_matrix = corr_matrix.loc[FEATURE_COLS, FEATURE_COLS]

print(f'Перевірка пар ознак з |r| > {CORR_THRESHOLD}:\n')
found_any = False
for i, f1 in enumerate(FEATURE_COLS):
    for f2 in FEATURE_COLS[i+1:]:
        if abs(feature_corr_matrix.loc[f1, f2]) > CORR_THRESHOLD:
            found_any = True
            keep = f1 if abs(corr_matrix.loc[f1, TARGET_COL]) >= abs(corr_matrix.loc[f2, TARGET_COL]) else f2
            drop = f2 if keep == f1 else f1
            drop_features.add(drop)
            print(f'  ⚠️  "{f1}" ↔ "{f2}" : r={feature_corr_matrix.loc[f1,f2]:.3f}')
            print(f'     Залишаємо: "{keep}" (r з таргетом = {corr_matrix.loc[keep, TARGET_COL]:.3f})')
            print(f'     Видаляємо: "{drop}"\n')

if not found_any:
    print('  ✅ Сильно корельованих пар не знайдено.')

selected_features = [f for f in sorted_features if f not in drop_features]
print(f'\n▶ Фінальні ознаки ({len(selected_features)} шт.): {selected_features}')

In [ ]:
X = df_clean[selected_features]
y = df_clean[TARGET_COL]

# Поділ 80/20 з фіксованим random_state=42 для відтворюваності
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'▶ Тренувальна вибірка: {X_train.shape[0]} зразків ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'▶ Тестова вибірка:     {X_test.shape[0]} зразків ({X_test.shape[0]/len(X)*100:.0f}%)')
print()
print('💡 random_state=42 гарантує однаковий поділ при кожному запуску — результати відтворювані.')

---
## 🔧 Допоміжні функції

In [ ]:
def compute_metrics(model, X_tr, X_te, y_tr, y_te, label=''):
    """
    Обчислює повний набір метрик для оцінки регресійної моделі.

    Метрики:
    - R²           : частка поясненої дисперсії (1.0 = ідеально)
    - MAE          : середня абсолютна похибка (в одиницях цільової змінної)
    - MSLE         : чутливий до відносних відхилень; не застосовний після Yeo-Johnson
    - MaxErr       : максимальна похибка (найгірший окремий прогноз)
    - D²AbsErr     : аналог R² для MAE-оптимізації
    """
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)

    def safe_msle(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        if np.any(y_true <= -1) or np.any(y_pred <= -1):
            return float('nan')  # Yeo-Johnson дає від'ємні значення — MSLE не застосовний
        y_pred_clip = np.clip(y_pred, 0, None)
        return round(mean_squared_log_error(y_true, y_pred_clip), 4)

    return {
        'Модель': label,
        'R² (train)':        round(model.score(X_tr, y_tr), 4),
        'R² (test)':         round(model.score(X_te, y_te), 4),
        'MAE (train)':       round(mean_absolute_error(y_tr, y_tr_pred), 4),
        'MAE (test)':        round(mean_absolute_error(y_te, y_te_pred), 4),
        'MSLE (train)':      safe_msle(y_tr, y_tr_pred),
        'MSLE (test)':       safe_msle(y_te, y_te_pred),
        'MaxErr (train)':    round(max_error(y_tr, y_tr_pred), 4),
        'MaxErr (test)':     round(max_error(y_te, y_te_pred), 4),
        'D²AbsErr (train)':  round(d2_absolute_error_score(y_tr, y_tr_pred), 4),
        'D²AbsErr (test)':   round(d2_absolute_error_score(y_te, y_te_pred), 4),
    }


def plot_regression(models_info, X_tr, X_te, y_tr, y_te, best_feat, title='Регресія'):
    """
    Scatter plot фактичних значень + лінії прогнозів для порівняння моделей.
    Проекція на вісь 'best_feat' для 2D-відображення.
    """
    feat_idx = list(X_tr.columns).index(best_feat)
    x_tr_feat = X_tr.iloc[:, feat_idx].values
    x_te_feat = X_te.iloc[:, feat_idx].values

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(x_tr_feat, y_tr, alpha=0.25, s=8, color='steelblue', label='Train (факт)')
    ax.scatter(x_te_feat, y_te, alpha=0.25, s=8, color='darkorange', label='Test (факт)')

    pred_colors = ['blue', 'red', 'green', 'purple', 'brown']
    line_styles  = ['-', '--', '-.', ':', '-']

    for i, info in enumerate(models_info):
        model = info['model']
        color = pred_colors[i % len(pred_colors)]
        ls    = line_styles[i % len(line_styles)]
        sort_tr = np.argsort(x_tr_feat)
        sort_te = np.argsort(x_te_feat)
        ax.plot(x_tr_feat[sort_tr], model.predict(X_tr.iloc[sort_tr]),
                color=color, ls=ls, lw=1.8, label=f"{info['label']} — train")
        ax.plot(x_te_feat[sort_te], model.predict(X_te.iloc[sort_te]),
                color=color, ls=':', lw=1.5, label=f"{info['label']} — test")

    ax.set_xlabel(best_feat, fontsize=12)
    ax.set_ylabel(TARGET_COL, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


all_metrics = []  # збираємо метрики всіх моделей
print('✅ Допоміжні функції визначено')

---
## 📐 ЗАВДАННЯ 2. ПОБУДОВА РЕГРЕСІЙНИХ МОДЕЛЕЙ

### 2.1 Проста лінійна регресія (Baseline)

**Модель:** `LinearRegression` з sklearn — МНК (метод найменших квадратів).

**Роль:** базова модель (baseline), з якою порівнюємо всі інші варіанти.

**Що очікуємо:** помірний R², оскільки залежність між доходом і ціною нелінійна.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

metrics_21 = compute_metrics(lr, X_train, X_test, y_train, y_test, label='LR (2.1)')
all_metrics.append(metrics_21)

# Таблиця метрик
df_metrics_21 = pd.DataFrame([metrics_21]).set_index('Модель')
print('📊 Метрики моделі 2.1:')
display(df_metrics_21.T)

In [ ]:
# Коефіцієнти моделі
coef_df = pd.DataFrame({
    'Ознака': selected_features,
    'Коефіцієнт': lr.coef_
}).sort_values('Коефіцієнт', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['seagreen' if c > 0 else 'tomato' for c in coef_df['Коефіцієнт']]
ax.barh(coef_df['Ознака'], coef_df['Коефіцієнт'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Коефіцієнти лінійної регресії (2.1)', fontsize=13)
ax.set_xlabel('Значення коефіцієнта')
plt.tight_layout()
plt.show()

print(f'Intercept (вільний член): {lr.intercept_:.4f}')

In [ ]:
plot_regression(
    [{'label': 'LinearRegression', 'model': lr}],
    X_train, X_test, y_train, y_test,
    BEST_FEATURE,
    title='2.1 Лінійна регресія (Baseline)'
)

r2_tr = metrics_21['R² (train)']
r2_te = metrics_21['R² (test)']
gap = abs(r2_tr - r2_te)

print('\n📝 Висновок (2.1):')
print(f'   R²(train) = {r2_tr},  R²(test) = {r2_te},  різниця = {gap:.4f}')
if gap < 0.03:
    print('   ✅ Модель добре узагальнюється — переоснащення відсутнє.')
else:
    print('   ⚠️  Є ознаки переоснащення або нелінійності — варто спробувати перетворення.')

### 2.2 Лінійна регресія з перетворенням Yeo-Johnson

**Мета:** наблизити розподіл ознак і цільової змінної до нормального — лінійна регресія краще працює на таких даних.

**Метод:** `PowerTransformer(method='yeo-johnson', standardize=True)`

> 💡 **Чому Yeo-Johnson, а не Box-Cox?**  
> Box-Cox вимагає строго додатних значень. Yeo-Johnson працює і з нулями, і з від'ємними значеннями — що важливо для нормалізованих ознак типу `Longitude`.

> ⚠️ **Важливо:** після Yeo-Johnson значення стандартизовані → можуть бути від'ємними → **MSLE не застосовний** і повертає `NaN`.

In [ ]:
# Трансформація ознак
pt = PowerTransformer(method='yeo-johnson', standardize=True)
X_train_pt = pd.DataFrame(pt.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_pt  = pd.DataFrame(pt.transform(X_test),      columns=X_test.columns,  index=X_test.index)

# Трансформація цільової змінної
pt_y = PowerTransformer(method='yeo-johnson', standardize=True)
y_train_pt = pd.Series(pt_y.fit_transform(y_train.values.reshape(-1,1)).ravel(), index=y_train.index)
y_test_pt  = pd.Series(pt_y.transform(y_test.values.reshape(-1,1)).ravel(),      index=y_test.index)

# Порівняння розподілів до/після
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
features_to_show = selected_features[:3]

for j, feat in enumerate(features_to_show):
    axes[0, j].hist(X_train[feat], bins=40, color='steelblue', alpha=0.8, edgecolor='white')
    axes[0, j].set_title(f'{feat}\n(до трансформації)')
    axes[1, j].hist(X_train_pt[feat], bins=40, color='seagreen', alpha=0.8, edgecolor='white')
    axes[1, j].set_title(f'{feat}\n(після Yeo-Johnson)')

plt.suptitle('Розподіл ознак до та після перетворення Yeo-Johnson', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
lr_pt = LinearRegression()
lr_pt.fit(X_train_pt, y_train_pt)

metrics_22 = compute_metrics(lr_pt, X_train_pt, X_test_pt, y_train_pt, y_test_pt,
                              label='LR+PowerTransform (2.2)')
all_metrics.append(metrics_22)

df_metrics_22 = pd.DataFrame([metrics_22]).set_index('Модель')
print('📊 Метрики моделі 2.2:')
display(df_metrics_22.T)

In [ ]:
plot_regression(
    [{'label': 'LR+PowerTransform', 'model': lr_pt}],
    X_train_pt, X_test_pt, y_train_pt, y_test_pt,
    BEST_FEATURE,
    title='2.2 Лінійна регресія з перетворенням Yeo-Johnson'
)

print('\n📝 Порівняння з baseline:')
print(f"   2.1 LR       R²(test) = {metrics_21['R² (test)']}")
print(f"   2.2 LR+PT    R²(test) = {metrics_22['R² (test)']}")
delta = metrics_22['R² (test)'] - metrics_21['R² (test)']
print(f"   Зміна: {delta:+.4f}")

### 2.3 Гребенева регресія (Ridge)

**Ідея:** додаємо до функції втрат L2-штраф `α·||w||²`, що запобігає великим коефіцієнтам.

$$\text{Ridge: } \min_{w} \|Xw - y\|^2_2 + \alpha \|w\|^2_2$$

**Вибір значень α:**

| α | Ефект |
|---|-------|
| **0.01** | Майже без регуляризації → близько до звичайної LR |
| **1.0** | Стандартне значення (баланс між точністю і регуляризацією) |
| **100.0** | Сильна регуляризація → коефіцієнти сильно стискаються до нуля |

In [ ]:
alpha_values = [1.0, 0.01, 100.0]
ridge_models = {}

for alpha in alpha_values:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    ridge_models[alpha] = ridge
    m = compute_metrics(ridge, X_train, X_test, y_train, y_test, label=f'Ridge α={alpha} (2.3)')
    all_metrics.append(m)

df_metrics_23 = pd.DataFrame(
    [compute_metrics(ridge_models[a], X_train, X_test, y_train, y_test, label=f'α={a}')
     for a in alpha_values]
).set_index('Модель')

print('📊 Порівняння Ridge з різними α:')
display(df_metrics_23.T)

In [ ]:
# Порівняння коефіцієнтів при різних α
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(selected_features))
width = 0.25
colors_bar = ['royalblue', 'tomato', 'seagreen']

for i, alpha in enumerate(alpha_values):
    ax.bar(x + i*width, ridge_models[alpha].coef_, width,
           label=f'α={alpha}', color=colors_bar[i], alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(selected_features, rotation=20)
ax.set_ylabel('Значення коефіцієнта')
ax.set_title('Ridge: порівняння коефіцієнтів при різних α', fontsize=13)
ax.legend()
ax.axhline(0, color='black', lw=0.8)
plt.tight_layout()
plt.show()

print('💡 При α=100 коефіцієнти значно зменшуються — сильна регуляризація.')

In [ ]:
plot_regression(
    [{'label': f'Ridge α={a}', 'model': ridge_models[a]} for a in alpha_values],
    X_train, X_test, y_train, y_test,
    BEST_FEATURE,
    title='2.3 Гребенева регресія (Ridge) — порівняння α'
)

### 2.4 Ridge + перетворення Yeo-Johnson

**Поєднання:** регуляризація (Ridge) + нормалізація розподілу (Yeo-Johnson).

**Очікування:** Ridge особливо корисний після трансформації, бо стандартизовані ознаки на одній шкалі — штраф застосовується рівномірно.

In [ ]:
ridge_pt_models = {}

for alpha in alpha_values:
    ridge_pt = Ridge(alpha=alpha)
    ridge_pt.fit(X_train_pt, y_train_pt)
    ridge_pt_models[alpha] = ridge_pt
    m = compute_metrics(ridge_pt, X_train_pt, X_test_pt, y_train_pt, y_test_pt,
                        label=f'Ridge+PT α={alpha} (2.4)')
    all_metrics.append(m)

df_metrics_24 = pd.DataFrame(
    [compute_metrics(ridge_pt_models[a], X_train_pt, X_test_pt, y_train_pt, y_test_pt,
                     label=f'Ridge+PT α={a}')
     for a in alpha_values]
).set_index('Модель')

print('📊 Порівняння Ridge+PT з різними α:')
display(df_metrics_24.T)

In [ ]:
plot_regression(
    [{'label': f'Ridge+PT α={a}', 'model': ridge_pt_models[a]} for a in alpha_values],
    X_train_pt, X_test_pt, y_train_pt, y_test_pt,
    BEST_FEATURE,
    title='2.4 Ridge + Yeo-Johnson — порівняння α'
)

---
## 📊 2.5 Зведене порівняння всіх моделей

In [ ]:
df_all = pd.DataFrame(all_metrics).set_index('Модель')

# Розділяємо на абсолютні (менше = краще) та відносні (більше = краще)
abs_cols = [c for c in df_all.columns if any(k in c for k in ['MAE', 'MSLE', 'MaxErr'])]
rel_cols = [c for c in df_all.columns if any(k in c for k in ['R²', 'D²'])]

df_absolute = df_all[abs_cols].copy()
df_relative = df_all[rel_cols].copy()

print('📊 Абсолютні метрики (менше = краще):')
styled_abs = df_absolute.style.background_gradient(cmap='RdYlGn_r', axis=0).format('{:.4f}')
display(styled_abs)

In [ ]:
print('📊 Відносні метрики (більше = краще):')
styled_rel = df_relative.style.background_gradient(cmap='RdYlGn', axis=0).format('{:.4f}')
display(styled_rel)

In [ ]:
# Графік R² (test) для всіх моделей
fig, ax = plt.subplots(figsize=(12, 5))
r2_test = df_all['R² (test)']
colors_bar = ['steelblue' if 'PT' not in idx else 'seagreen' for idx in r2_test.index]

bars = ax.barh(r2_test.index, r2_test.values, color=colors_bar, edgecolor='white', height=0.6)
ax.axvline(r2_test.max(), color='red', linestyle='--', alpha=0.7, label=f'Макс. R²={r2_test.max():.4f}')

for bar, val in zip(bars, r2_test.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', fontsize=9)

ax.set_xlabel('R² (test)', fontsize=12)
ax.set_title('Порівняння R² на тестовій вибірці', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xlim(0, r2_test.max() + 0.05)
plt.tight_layout()
plt.show()

In [ ]:
# Графік MAE (test) для всіх моделей
fig, ax = plt.subplots(figsize=(12, 5))
mae_test = df_all['MAE (test)']
colors_bar2 = ['tomato' if 'PT' not in idx else 'darkorange' for idx in mae_test.index]

bars2 = ax.barh(mae_test.index, mae_test.values, color=colors_bar2, edgecolor='white', height=0.6)
ax.axvline(mae_test.min(), color='green', linestyle='--', alpha=0.7, label=f'Мін. MAE={mae_test.min():.4f}')

for bar, val in zip(bars2, mae_test.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', fontsize=9)

ax.set_xlabel('MAE (test)', fontsize=12)
ax.set_title('Порівняння MAE на тестовій вибірці (менше = краще)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 🏁 Загальні висновки

| Аспект | Висновок |
|--------|----------|
| **Найважливіша ознака** | `MedInc` — медіанний дохід найсильніше визначає вартість житла |
| **Yeo-Johnson** | Покращує або погіршує R² в залежності від нелінійності; MSLE стає недоступним |
| **Ridge vs LR** | Різниця незначна на цьому датасеті — мультиколінеарність невисока після відбору ознак |
| **Вибір α** | α=0.01 ≈ LR; α=100 — надмірна регуляризація, може знизити якість |
| **Переоснащення** | Мале (|R²(train) − R²(test)| < 0.03) — лінійна модель не схильна до overfit на цих даних |

In [ ]:
# Фінальна таблиця — топ-3 моделі за R²(test)
top3 = df_all['R² (test)'].sort_values(ascending=False).head(3)
print('🏆 Топ-3 моделі за R²(test):')
for rank, (name, score) in enumerate(top3.items(), 1):
    medal = ['🥇','🥈','🥉'][rank-1]
    print(f'  {medal} {rank}. {name}: R²={score:.4f}')

best_model_name = top3.index[0]
best_mae = df_all.loc[best_model_name, 'MAE (test)']
print(f'\n📌 Найкраща модель: {best_model_name}')
print(f'   MAE = {best_mae:.4f} (×$100 000 = ±${best_mae*100_000:,.0f} середня похибка прогнозу)')